# Research/experiment notes

## Scope

We focus on learned algorithmic behaviour. In particular, we investigate how a fixed, single-layer GRU learns a simple arithmetic operation from sequences, how faithfully its learned computation respects the mathematical properties of counting, and how that computation changes when the statistical or structural properties of the input sequences differ from training.

## Literature

Three papers are especially interesting and serve as motivation for the counting experiment:

Weiss, Goldberg & Yahav (2018) examine computational limitations of finite-precision recurrent architectures and identify important differences between GRUs and recurrent networks with more explicit counting capabilities. Their results motivate studying the limits of GRU state representations, although our finite-horizon count-regression problem is not identical to their formal-language setting. 


Delétang et al., Neural Networks and the Chomsky Hierarchy study whether neural networks can generalise algorithmic behaviour beyond their training regimes. Their experiments reinforce the importance of distinguishing fitting training examples from learning computations that remain valid at greater lengths. Again, their language-recognition tasks are not interchangeable with ours. 


Geirhos et al. (2020), Shortcut Learning in Deep Neural Networks provide a conceptual framework for testing whether a model exploits predictive statistical features that cease to be useful under changed conditions. This motivates our investigation of the correlation between sequence length and expected count. It does not establish that our GRU actually uses that shortcut.

## 1. Research motivation and scientific scope

### 1.1 Motivation

Recurrent neural networks process sequences through repeated transformations of an internal hidden state. Their behaviour therefore depends not only on the parameters learned during training, but also on how information is represented, transformed, retained, and potentially lost as successive inputs are processed.

Gated recurrent units (GRUs) introduce reset and update gates to regulate these transformations (Cho et al., 2014). Such gating permits the network to learn dependencies across multiple timesteps, but does not guarantee that the resulting computation respects the mathematical structure of the task being learned.

This distinction becomes particularly important in algorithmic and mathematical generalisation.

A model can achieve low prediction error on sequences drawn from its training distribution without necessarily implementing a computation that remains correct across different sequence lengths, token arrangements, or statistical distributions.

Research on the computational capabilities of finite-precision recurrent networks highlights limitations of bounded recurrent representations for certain counting tasks (Weiss, Goldberg & Yahav, 2018). Broader studies of neural networks on formal-language tasks likewise demonstrate that strong in-distribution performance need not imply successful extrapolation (Delétang et al., 2023).

A related concern is shortcut learning: models may exploit predictive statistical relationships present in their training environments, even when those relationships are insufficient to support robust performance under distribution shift (Geirhos et al., 2020).

These papers motivate the following counting experiment using a single-layer, explicitly implemented GRU.

### 1.2 Why token counting?

We study the simple arithmetic task

$$
f^*(x_1,\ldots,x_T)=\sum_{t=1}^{T}x_t,
\qquad x_t\in{0,1}.
$$

The function is mathematically elementary, and its output can be determined exactly for any finite binary sequence.

Nevertheless, learning to implement this function recurrently requires the model to develop a computation that is sufficiently sensitive to one-valued tokens, sufficiently insensitive to zero-valued tokens, and sufficiently stable across repeated state transitions.

Counting is therefore a useful minimal setting in which to distinguish:

* Statistical prediction from algorithmic computation.

* In-distribution task learning from length extrapolation.

* Correct output predictions from particular hidden-state mechanisms.

* Mathematical architectural constraints from limitations of training or optimisation.

The simplicity of the target function makes behavioural failures relatively easy to define and measure. It does not make their causes immediately obvious.

## 2. Experiments
### 2.1 An important architectural bound

Since $z_{t,i}\in(0,1)$ and $\tilde h_{t,i}\in(-1,1)$, each ordinary state update is a convex combination of the previous coordinate and a bounded candidate.

Starting from $h_0=0$, induction gives

$$
|h_{t,i}|\leq1.
$$

Therefore,

$$
\|h_t\|_2\leq\sqrt K
$$

and

$$
|\hat Y|\leq\|w_{\mathrm{out}}\|_1+|b_{\mathrm{out}}|.
$$

Thus, for fixed finite parameters, this architecture cannot implement exact unbounded count regression for arbitrarily large counts.

## 3. Statistical verification of the baseline

### 3.1 Training distribution

Our initial training distribution is defined by

$$
T\sim\operatorname{Uniform}\{5,6,\ldots,20\},
$$

followed by independent binary token sampling:

$$
X_t\mid T\overset{\mathrm{iid}}{\sim}\operatorname{Bernoulli}(p),
\qquad p=0.5.
$$

The final target is

$$
\boxed{Y=\sum_{t=1}^{T}X_t.}
$$

Conditional on length,

$$
Y\mid T\sim\operatorname{Binomial}(T,0.5),
$$

with

$$
\mathbb E[Y\mid T]=\frac T2,
\qquad
\operatorname{Var}(Y\mid T)=\frac T4.
$$

Training minimises the empirical final-count MSE:

$$
\mathcal L(\theta)
=
\frac1B\sum_{b=1}^{B}
\left(f_\theta(X^{(b)})-Y^{(b)}\right)^2.
$$

Intermediate cumulative counts

$$
c_t=\sum_{j=1}^{t}x_j
$$

are mathematically defined and available for analysis, but are **not separately supervised during baseline training**.

This distinction is important. The network is only required by the training objective to predict the count at the final timestep. An accurate final prediction does not imply that every intermediate hidden state represents the true prefix count in a directly readable form.

#### Reference A — Constant predictor

Without observing either the sequence or its length, the optimal constant prediction under squared loss is

$$
\hat Y_{\mathrm{constant}}=\mathbb E[Y].
$$

Since $\mathbb E[T]=12.5$,

$$
\mathbb E[Y]=0.5(12.5)=6.25.
$$

Using the law of total variance,

$$
\begin{aligned}
\operatorname{Var}(Y)
&=\mathbb E[\operatorname{Var}(Y\mid T)]
+\operatorname{Var}(\mathbb E[Y\mid T])\\
&=\frac{\mathbb E[T]}4+\frac{\operatorname{Var}(T)}4\\
&=3.125+5.3125\\
&=\boxed{8.4375}.
\end{aligned}
$$

Therefore, the constant predictor's population MSE is 8.4375.

#### Reference B — Length-only predictor

If the predictor knows $T$ but does not inspect token values, its Bayes-optimal prediction under squared loss is

$$
\boxed{\hat Y_{\mathrm{length}}=\mathbb E[Y\mid T]=T/2.}
$$

Its conditional MSE is $T/4$, so

$$
\mathbb E[(Y-T/2)^2]
=\frac{\mathbb E[T]}4
=\boxed{3.125}.
$$

This predictor is statistically legitimate under the training distribution, despite not counting individual tokens.

It is therefore a necessary reference for evaluating how much the GRU learns beyond sequence length.

#### Reference C — Exact counting oracle

The exact target function is

$$
\hat Y_{\mathrm{oracle}}
=\sum_{t=1}^{T}X_t=Y.
$$

Its MSE is zero.

It is not a learned model; it serves as an exact computational reference and a check on target generation.

| Predictor | Information used | Population MSE |
|---|---|---:|
| Constant $6.25$ | Neither tokens nor length | 8.4375 |
| Length-only $T/2$ | Length | 3.125 |
| Counting oracle | Full token sequence | 0 |
| Trained GRU | Learned function of the sequence | To be measured |

These are population expectations. Empirical estimates from finite samples will vary.

A GRU that performs better than $T/2$ has learned predictive information beyond length alone. That result does not prove the absence of length dependence or establish an exact counting representation.

### 3.2 Research question

**Baseline: Can the verified GRU implementation learn the finite counting task reliably throughout its specified in-distribution training regime?**

### 3.2 Computational integrity

Before training, we confirm that the implementation satisfies its mathematical contract. Verifications are maths_framework.ipynb

### 3.3 Baseline protocol decisions to freeze before execution

The architecture and statistical task are already specified. The following remaining experimental settings must be recorded before the first new run:

| Item | Proposed or existing setting |
|---|---|
| Hidden dimension | 16 |
| Training lengths | 5–20 |
| Bernoulli probability | 0.5 |
| Batch size | 64 |
| Optimiser | Adam |
| Learning rate | $10^{-3}$ |
| GRU weight initialisation | $\sigma=0.01$ |
| Weight decay | None |
| Model seed | 42 |
| Training-data seed | 43 |
| Validation-data seed | 44 |
| Maximum updates | To be frozen before execution; 10,000 is a candidate informed by the pilot |
| Model-selection rule | Final checkpoint under the present implementation, unless prospectively revised |
| Final ID evaluation | Independent held-out Bernoulli batches, separately reported for every $T=5,\ldots,20$ |

### possible future directions
Freivalds and Liepins (2018) demonstrate that hard nonlinearities combined with saturation costs improve training and length generalisation in their modified Neural GPU architecture, and propose possible extensions to ordinary GRUs and LSTMs. These results motivate a separate controlled investigation of activation functions in standard GRUs. Whether any benefits arise from improved optimisation, altered gate behaviour, more stable recurrent-state dynamics, or other mechanisms represent possible interesting questions. Presently, conventional sigmoid and tanh activations are retained to avoid introducing additional experimental variables.

Reference: Freivalds, K., & Liepins, R. (2018). Improving the Neural GPU Architecture for Algorithm Learning. arXiv:1702.08727.